# Laser Printing System — Interface Walkthrough

This notebook is a **training / reference guide** for the three layers of the control API. Run each cell in order. Cells marked **`HARDWARE`** actually talk to the devices — read the safety comment before running them.

### Layers
- **Layer 0** (`laser_printing.hardware`) — raw transports. `LaserHttp`, `StageTcp`. No policy, no validation.
- **Layer 1** (`laser_printing.controllers`) — primitive actions. `LaserController` (`on`/`off`/`set_attenuator`…), `StageController` (`move_absolute`/`move_relative`/`jog`…). Safety clamps, state cache, context managers.
- **Layer 2** (`laser_printing.controllers.sync`) — coordinated actions. `PrintSynchronizer.execute_path` + strategy selection.

### Path convention (every generator in this package follows it)
```
path[i] = (point_i, laser_during_move_arriving_at_point_i)
```
`path[0]` is the initial reposition; its laser flag is ignored. Everything downstream (`single_line`, `lines_with_pitch`, STL pipeline, `execute_path`) uses this convention.

### Home position
The stage always homes at `[0, 0, 0] mm` — before *and* after every experiment. `StageController` enforces this on `connect()` and `disconnect()`.

## 1. Config & imports

In [ ]:
from laser_printing.config import load_config

config = load_config()
print(list(config.keys()))
print('laser:', config['laser'])
print('stage:', config['stage'])
print('sync :', config['synchronization'])

## 2. Layer 0 — raw transports (no policy)

These modules are what Layer 1 is built on. Use them directly only when you need to poke at a specific firmware endpoint that Layer 1 doesn't expose.

In [ ]:
# HARDWARE: read-only HTTP call to the laser
from laser_printing.hardware import LaserHttp

with LaserHttp() as http:
    status = http.get_status()

# Only print a handful of interesting keys
for k in ['IsOutputEnabled', 'ActualStateName', 'TargetAttenuatorPercentage',
          'TargetPpDivider', 'ActualOutputPower', 'ActualOutputFrequency']:
    print(f'  {k:32s} {status[k]!r}')

In [ ]:
# HARDWARE: open the stage TCP, read positions, close. No motion.
from laser_printing.hardware import StageTcp

with StageTcp() as s:
    pos = [s.get_fposition(ax) for ax in (0, 1, 2)]
print('stage position (mm):', pos)

## 3. Layer 1 — primitive actions

`LaserController` polls the status until the laser actually changed state (our firmware takes ~150–300 ms to physically toggle after the POST returns). `StageController` clamps every move to `max_step_mm` so a runaway coordinate can't slam the stage.

In [ ]:
# Safe to *construct* anywhere - does not touch the hardware.
from laser_printing.controllers import LaserController, StageController

laser = LaserController.from_config(config['laser'])
stage = StageController.from_config(config['stage'])
print(laser, stage)
print('laser.is_on (before connect):', laser.is_on)   # None - unknown
print('stage.max_step_mm:', stage.max_step_mm)
print('stage.range_mm :', stage.range_mm)

In [ ]:
# HARDWARE: LaserController - context manager version.
# On __enter__ the current hardware state is loaded into the cache.
# On __exit__ the laser is turned OFF.
with LaserController.from_config(config['laser']) as laser:
    print('initial state: enabled=%s atten=%s div=%s' % (
        laser.is_on, laser.attenuator_pct, laser.pp_divider))
    metrics_off = laser.off()
    print('off: post=%.1f ms, settle=%.1f ms, polls=%d'
          % (metrics_off.post_ms, metrics_off.settle_ms, metrics_off.poll_count))
    metrics_on = laser.on()
    print('on : post=%.1f ms, settle=%.1f ms, polls=%d'
          % (metrics_on.post_ms, metrics_on.settle_ms, metrics_on.poll_count))
    # laser turns OFF on context exit

In [ ]:
# HARDWARE: StageController primitives.
# ENTER moves to [0,0,0]. EXIT also moves to [0,0,0] before closing.
# Every move below is clamped by stage.max_step_mm (5 mm by default).
with StageController.from_config(config['stage']) as stage:
    print('start position:', stage.position())

    stage.set_velocity(1.0)                  # 1 mm/s on every axis
    stage.move_relative([0.05, 0, 0])         # +50 um on X
    print('after +X 50um :', stage.position())

    stage.jog(axis=1, distance_mm=-0.05)     # -50 um on Y via jog helper
    print('after jog Y -50um:', stage.position())

    stage.move_absolute([0, 0, 0])           # back to home
    print('after home      :', stage.position())
# on exit, stage homes again and closes TCP.

In [ ]:
# Safety clamp demonstration (no hardware required - the check runs
# before any command is sent).
from laser_printing.controllers import StageController, StageSafetyError

stage = StageController(max_step_mm=0.2)  # aggressive clamp
# We don't connect - just show the clamp math directly:
try:
    stage._check_step([0.0, 0.0, 0.0], [1.0, 0.0, 0.0], clamp_mm=None)
except StageSafetyError as e:
    print('Refused:', e)

## 4. Path generation (no hardware needed)

Generators produce the canonical path format. Use these to *plan* and *preview* an experiment without any hardware connected.

In [ ]:
from laser_printing.path_generation.coordinates import (
    single_line, lines_with_pitch, z_stacks_with_power)

# Single line with 1 print pass:
print(single_line(x_start=0, x_end=1, y=0, z=6, repetitions=1))
# Two back-and-forth passes (both laser-on):
print(single_line(x_start=0, x_end=1, y=0, z=6, repetitions=2))

In [ ]:
blocks = lines_with_pitch(
    x_start=0, x_end=2, y_start=0, y_pitch=0.1,
    z_focus=6, count=5, repetitions=1)
print('%d blocks total (1 home + 5 lines)' % len(blocks))
print('home block:', blocks[0])
print('first line :', blocks[1])

In [ ]:
# 3D preview
import matplotlib
matplotlib.use('inline')
from laser_printing.visualization.plotting import plot_path_3d

# Flatten blocks into one path (for visualization only - execution is
# done block-by-block so parameters can change between lines).
all_points = [pt for block in blocks for pt in block]
plot_path_3d(all_points, title='5 lines, 100 um pitch', show=True)

## 5. Layer 2 — synchronized printing

`PrintSynchronizer` picks a strategy and runs a full path.

- **`sequential`**: laser toggles between moves. Use for positioning tests.
- **`velocity_timed`** (production): non-blocking move + calibrated sleep — laser fires only during the constant-velocity coast window. Accounts for both accel time *and* the laser's ~200 ms physical settle time (the `laser_on_lead_s` / `laser_off_lead_s` config keys).

In [ ]:
# Construction only (no hardware). Inspect the calibration table and
# latency knobs.
from laser_printing.controllers.sync import PrintSynchronizer, CalibrationPoint, LaserLatency

# Build from config without connecting the hardware yet.
dummy_laser = LaserController.from_config(config['laser'])
dummy_stage = StageController.from_config(config['stage'])
sync = PrintSynchronizer.from_config(config, dummy_laser, dummy_stage)
print('strategy     :', sync._strategy)
print('calibration  :', sync._calibration)
print('laser leads  :', sync._lat)
print('accel params at 5 mm/s:', sync._get_accel_params(5.0))
print('accel params at 3 mm/s (interpolated):', sync._get_accel_params(3.0))

In [ ]:
# HARDWARE: end-to-end 2 mm line at 3 mm/s with `sequential` strategy
# (safer first test - no timing assumptions).
# Coordinates are all <= 2 mm from home so the default clamp is fine.
# The line will be printed at y=0.5 mm, z=0 mm.

with LaserController.from_config(config['laser']) as laser, \
     StageController.from_config(config['stage']) as stage:
    # Force sequential for the first run.
    sync = PrintSynchronizer(laser, stage, strategy='sequential')

    laser.set_attenuator(10)   # keep low for the first test
    laser.set_pp_divider(1)
    stage.set_velocity(3.0)

    # 2 mm line along X at (y=0.5, z=0); 1 print pass.
    path = single_line(x_start=-1.0, x_end=1.0, y=0.5, z=0.0, repetitions=1)
    sync.execute_path(path)

# Both controllers always clean up on context exit.

## 6. Where to go next

- **Calibrate**: run `experiment/motion_profiling.MotionProfiling` at the velocities you plan to use, then paste the numbers into `config/default.yaml` under `synchronization.calibration`.
- **Parameter sweep**: `experiment/parameter_sweep.ParameterSweep(mode='line_doe' | 'z_stack')` uses the Layer 2 synchronizer automatically.
- **STL printing**: `path_generation/stl.generate_from_stl(...)` produces a single long path; hand it to `PrintSynchronizer.execute_path`.
- **Add a new primitive**: write it in Layer 1 on top of the Layer 0 transport — no Layer 2 code should ever call SPiiPlus or `requests` directly.